# Fix: MVRV Trail Exiting at Loss

**Problem:** MVRV trail can activate when barely in profit, then exit at a loss.

**Example:**
- Entry: $49,022
- Peak: $50,983 (+4%)
- MVRV > 2.0 → Trail activates
- Price drops 25% from peak → Exit at $38,237
- Result: -22% loss!

**Fixes to Test:**
1. **Profit Gate:** Only activate trail if gain > X%
2. **Higher MVRV:** Use 2.25 or 2.5 instead of 2.0
3. **Tighter Trail:** 15% or 20% instead of 25%
4. **Trail from Entry:** When MVRV triggers, trail from entry price not peak

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("MVRV Trail Exit Fix Testing 🔧")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']

df_test = df[df.index >= '2018-12-15'].copy().dropna()
print(f"Data: {len(df_test)} rows")

In [ ]:
def sopr_rl_entry(df, rl_z_threshold=0.5):
    sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
    rl_signal = df['rl_zscore'] > rl_z_threshold
    combined = sopr_signal & rl_signal
    entries = combined & ~combined.shift(1).fillna(False)
    return entries

---
## Current Strategy (Baseline)

In [ ]:
def backtest_original(df, entries, mvrv_trigger=2.0, trailing_pct=0.25, stop_loss=0.20, max_hold_days=365):
    """Original strategy - trail from peak, no profit gate."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            # Original: activate trail when MVRV triggers (no profit check)
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Baseline results
entries = sopr_rl_entry(df_test, 0.5)
trades_baseline = backtest_original(df_test, entries)

# Find problem trades (MVRV trail exits at loss)
problem_trades = trades_baseline[(trades_baseline['exit_reason'] == 'mvrv_trail') & (trades_baseline['pnl_pct'] < 0)]

print("BASELINE: Original Strategy")
print("="*60)
print(f"Total trades: {len(trades_baseline)}")
print(f"Win rate: {(trades_baseline['pnl_pct'] > 0).mean()*100:.0f}%")
print(f"Total return: {((1+trades_baseline['pnl_pct']).prod()-1)*100:+.0f}%")
print(f"\n⚠️ Problem trades (MVRV trail exits at loss): {len(problem_trades)}")

if len(problem_trades) > 0:
    print(f"\nProblem trades:")
    for _, t in problem_trades.iterrows():
        print(f"  {t['entry_date'].date()}: {t['pnl_pct']*100:+.0f}%")

---
## Fix 1: Profit Gate

In [ ]:
def backtest_profit_gate(df, entries, mvrv_trigger=2.0, trailing_pct=0.25, stop_loss=0.20, 
                         max_hold_days=365, min_profit_to_trail=0.10):
    """Only activate trail if MVRV triggers AND we have min profit."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            gain_from_entry = (peak_price - entry_price) / entry_price
            
            # FIX: Only activate trail if MVRV triggers AND we have min profit
            if not trailing_active and current_mvrv >= mvrv_trigger and gain_from_entry >= min_profit_to_trail:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Test different profit gates
print("FIX 1: PROFIT GATE")
print("="*80)
print(f"{'Min Profit':<15} {'Trades':>10} {'Win Rate':>12} {'Total Ret':>12} {'Problem Exits':>15}")
print("-"*80)

profit_gate_results = []

for min_profit in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    trades = backtest_profit_gate(df_test, entries, min_profit_to_trail=min_profit)
    
    problem = len(trades[(trades['exit_reason'] == 'mvrv_trail') & (trades['pnl_pct'] < 0)])
    win_rate = (trades['pnl_pct'] > 0).mean() * 100
    total_ret = ((1 + trades['pnl_pct']).prod() - 1) * 100
    
    print(f"{min_profit*100:>10.0f}% {len(trades):>10} {win_rate:>11.0f}% {total_ret:>+11.0f}% {problem:>15}")
    
    profit_gate_results.append({
        'min_profit': min_profit,
        'trades': len(trades),
        'win_rate': win_rate,
        'total_return': total_ret,
        'problem_exits': problem,
        'trades_df': trades
    })

---
## Fix 2: Higher MVRV Trigger

In [ ]:
print("\nFIX 2: HIGHER MVRV TRIGGER")
print("="*80)
print(f"{'MVRV Trigger':<15} {'Trades':>10} {'Win Rate':>12} {'Total Ret':>12} {'Problem Exits':>15}")
print("-"*80)

mvrv_results = []

for mvrv_t in [1.75, 2.0, 2.25, 2.5, 2.75, 3.0]:
    trades = backtest_original(df_test, entries, mvrv_trigger=mvrv_t)
    
    problem = len(trades[(trades['exit_reason'] == 'mvrv_trail') & (trades['pnl_pct'] < 0)])
    win_rate = (trades['pnl_pct'] > 0).mean() * 100
    total_ret = ((1 + trades['pnl_pct']).prod() - 1) * 100
    
    print(f"MVRV > {mvrv_t:<8} {len(trades):>10} {win_rate:>11.0f}% {total_ret:>+11.0f}% {problem:>15}")
    
    mvrv_results.append({
        'mvrv_trigger': mvrv_t,
        'trades': len(trades),
        'win_rate': win_rate,
        'total_return': total_ret,
        'problem_exits': problem
    })

---
## Fix 3: Tighter Trail %

In [ ]:
print("\nFIX 3: TIGHTER TRAIL %")
print("="*80)
print(f"{'Trail %':<15} {'Trades':>10} {'Win Rate':>12} {'Total Ret':>12} {'Problem Exits':>15}")
print("-"*80)

trail_results = []

for trail in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]:
    trades = backtest_original(df_test, entries, trailing_pct=trail)
    
    problem = len(trades[(trades['exit_reason'] == 'mvrv_trail') & (trades['pnl_pct'] < 0)])
    win_rate = (trades['pnl_pct'] > 0).mean() * 100
    total_ret = ((1 + trades['pnl_pct']).prod() - 1) * 100
    
    print(f"{trail*100:>10.0f}% {len(trades):>10} {win_rate:>11.0f}% {total_ret:>+11.0f}% {problem:>15}")
    
    trail_results.append({
        'trail_pct': trail,
        'trades': len(trades),
        'win_rate': win_rate,
        'total_return': total_ret,
        'problem_exits': problem
    })

---
## Fix 4: Trail from Entry (Not Peak)

In [ ]:
def backtest_trail_from_entry(df, entries, mvrv_trigger=2.0, trailing_pct=0.25, stop_loss=0.20, max_hold_days=365):
    """When MVRV triggers, trail from ENTRY price not peak."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        trail_base_price = entry_price  # Trail from entry, not peak
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            # When MVRV triggers, lock in entry as trail base
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
                trail_base_price = entry_price  # Trail from entry
            
            if trailing_active:
                # Update trail base if we make new highs
                if current_price > trail_base_price:
                    trail_base_price = current_price
                
                trail_stop = trail_base_price * (1 - trailing_pct)
                # But never exit below entry!
                trail_stop = max(trail_stop, entry_price * 0.95)  # At least -5% from entry
                
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
print("\nFIX 4: TRAIL FROM ENTRY (with floor at -5%)")
print("="*80)

trades_fix4 = backtest_trail_from_entry(df_test, entries)

problem = len(trades_fix4[(trades_fix4['exit_reason'] == 'mvrv_trail') & (trades_fix4['pnl_pct'] < 0)])
win_rate = (trades_fix4['pnl_pct'] > 0).mean() * 100
total_ret = ((1 + trades_fix4['pnl_pct']).prod() - 1) * 100

print(f"Trades: {len(trades_fix4)}")
print(f"Win Rate: {win_rate:.0f}%")
print(f"Total Return: {total_ret:+.0f}%")
print(f"Problem Exits: {problem}")

---
## Walk-Forward Validation of Fixes

In [ ]:
def walk_forward(df, backtest_func, **kwargs):
    """Walk-forward validation."""
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end].copy()
        test_close = close.iloc[test_start:test_end]
        
        test_entries = sopr_rl_entry(test_df, 0.5)
        trades = backtest_func(test_df, test_entries, **kwargs)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    wf_df = pd.DataFrame(results)
    return wf_df['beat_hold'].mean(), (wf_df['strat_return'] - wf_df['hold_return']).mean()

In [ ]:
print("\nWALK-FORWARD VALIDATION")
print("="*80)
print(f"{'Strategy':<40} {'Beat Rate':>15} {'Avg Excess':>15}")
print("-"*80)

# Baseline
br, ae = walk_forward(df_test, backtest_original)
print(f"{'Original (MVRV>2.0, 25% trail)':<40} {br*100:>14.0f}% {ae*100:>+14.1f}%")
baseline_br = br

# Best profit gate
for min_p in [0.10, 0.15, 0.20]:
    br, ae = walk_forward(df_test, backtest_profit_gate, min_profit_to_trail=min_p)
    marker = "✅" if br > baseline_br else ""
    print(f"{'Profit Gate ' + str(int(min_p*100)) + '%':<40} {br*100:>14.0f}% {ae*100:>+14.1f}% {marker}")

# Higher MVRV
for mvrv_t in [2.25, 2.5]:
    br, ae = walk_forward(df_test, backtest_original, mvrv_trigger=mvrv_t)
    marker = "✅" if br > baseline_br else ""
    print(f"{'MVRV > ' + str(mvrv_t):<40} {br*100:>14.0f}% {ae*100:>+14.1f}% {marker}")

# Tighter trail
for trail in [0.15, 0.20]:
    br, ae = walk_forward(df_test, backtest_original, trailing_pct=trail)
    marker = "✅" if br > baseline_br else ""
    print(f"{'Trail ' + str(int(trail*100)) + '%':<40} {br*100:>14.0f}% {ae*100:>+14.1f}% {marker}")

# Trail from entry
br, ae = walk_forward(df_test, backtest_trail_from_entry)
marker = "✅" if br > baseline_br else ""
print(f"{'Trail from Entry (floor -5%)':<40} {br*100:>14.0f}% {ae*100:>+14.1f}% {marker}")

---
## Combined Fix: Best Parameters

In [ ]:
# Test combinations
print("\nCOMBINED FIX SEARCH")
print("="*100)
print(f"{'MVRV':>8} {'Trail':>8} {'Profit Gate':>12} {'Beat Rate':>12} {'Avg Excess':>12}")
print("-"*100)

best_br = 0
best_config = None

for mvrv_t in [2.0, 2.25, 2.5]:
    for trail in [0.20, 0.25, 0.30]:
        for min_p in [0.0, 0.10, 0.20]:
            br, ae = walk_forward(df_test, backtest_profit_gate, 
                                  mvrv_trigger=mvrv_t, trailing_pct=trail, min_profit_to_trail=min_p)
            
            marker = "⭐" if br > best_br else ""
            print(f"{mvrv_t:>8} {trail*100:>7.0f}% {min_p*100:>11.0f}% {br*100:>11.0f}% {ae*100:>+11.1f}% {marker}")
            
            if br > best_br:
                best_br = br
                best_config = {'mvrv': mvrv_t, 'trail': trail, 'min_profit': min_p, 'beat_rate': br, 'avg_excess': ae}

In [ ]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print(f"\n📊 ORIGINAL STRATEGY")
print(f"   MVRV > 2.0, Trail 25%, No profit gate")
print(f"   Beat rate: {baseline_br*100:.0f}%")
print(f"   Problem: {len(problem_trades)} trades exit via MVRV trail at a loss")

if best_config:
    print(f"\n📊 BEST FIXED STRATEGY")
    print(f"   MVRV > {best_config['mvrv']}, Trail {best_config['trail']*100:.0f}%, Profit gate {best_config['min_profit']*100:.0f}%")
    print(f"   Beat rate: {best_config['beat_rate']*100:.0f}%")
    print(f"   Improvement: {(best_config['beat_rate'] - baseline_br)*100:+.0f}%")

print(f"\n🎯 VERDICT:")
if best_config and best_config['beat_rate'] > baseline_br:
    print(f"   ✅ Fix improves strategy!")
    print(f"   → Update STRAT-002 with new parameters")
else:
    print(f"   ⚠️ Fix doesn't improve walk-forward beat rate")
    print(f"   → Problem trades may be acceptable cost of the strategy")

print("\n" + "="*80)

In [ ]:
# Save results
import json

results = {
    'baseline_beat_rate': baseline_br,
    'problem_trades_count': len(problem_trades),
    'best_config': best_config,
    'profit_gate_results': [{k:v for k,v in r.items() if k != 'trades_df'} for r in profit_gate_results],
    'mvrv_results': mvrv_results,
    'trail_results': trail_results
}

with open('../data/mvrv_trail_fix_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

print("Saved to ../data/mvrv_trail_fix_results.json")